# SECtion 6 get Link

In [ ]:
import requests
import os

def get_insect_images(scientific_name, limit=50):
    # 1. ค้นหา ID ของแมลงจากชื่อ
    species_url = f"https://api.gbif.org/v1/species/match?name={scientific_name}"
    species_id = requests.get(species_url).json().get('usageKey')

    if not species_id:
        return "Not Found"

    # 2. ค้นหาตำแหน่งที่มีรูปภาพ
    occ_url = f"https://api.gbif.org/v1/occurrence/search?speciesKey={species_id}&mediaType=StillImage&limit={limit}"
    results = requests.get(occ_url).json().get('results', [])

    # 3. ดึง URL ของภาพ
    image_urls = []
    for res in results:
        for media in res.get('media', []):
            if media.get('type') == 'StillImage':
                image_urls.append(media.get('identifier'))

    return image_urls

# ตัวอย่าง: ดึงภาพเพลี้ยกระโดดสีน้ำตาล (Nilaparvata lugens)
urls = get_insect_images("Stenchaetothrips biformis", limit=100)
print(f"พบภาพทั้งหมด {len(urls)} ภาพ")
print(urls)

# Section 7: Retreive URL for each category

In [ ]:
import pandas as pd
import requests
import os
from google.colab import drive

# 1. เชื่อมต่อกับ Google Drive
# drive.mount('/content/drive')

# 2. กำหนดพาธไฟล์ CSV และโฟลเดอร์ปลายทาง
csv_path = '/content/drive/MyDrive/Colab Notebooks/insect-data/gbif_insect_images.csv'
base_download_path = '/content/drive/MyDrive/Colab Notebooks/insect-data/downloaded_images/'

# 3. อ่านไฟล์ CSV
df = pd.read_csv(csv_path)

# 4. ฟังก์ชันสำหรับดาวน์โหลดรูปภาพ
def download_images(dataframe, save_root):
    # สร้างโฟลเดอร์หลักถ้ายังไม่มี
    if not os.path.exists(save_root):
        os.makedirs(save_root)
        print(f"สร้างโฟลเดอร์หลัก: {save_root}")

    print("เริ่มการดาวน์โหลด...")

    for index, row in dataframe.iterrows():
        # insect_name = str(row['Insect Name']).replace(" ", "_") # แทนที่ช่องว่างด้วย _ เพื่อชื่อโฟลเดอร์ที่ปลอดภัย
        insect_name = str(row['Insect Name Thai']).replace(" ", "_") # แทนที่ช่องว่างด้วย _ เพื่อชื่อโฟลเดอร์ที่ปลอดภัย
        image_url = row['Image URL']

        # สร้างโฟลเดอร์เฉพาะชนิดแมลง
        insect_folder = os.path.join(save_root, insect_name)
        if not os.path.exists(insect_folder):
            os.makedirs(insect_folder)

        # ตั้งชื่อไฟล์รูปภาพ (ใช้ index เพื่อไม่ให้ชื่อซ้ำ)
        file_extension = ".jpg" # ส่วนใหญ่จาก GBIF จะเป็น jpg
        file_name = f"{insect_name}_{index:04d}{file_extension}"
        file_path = os.path.join(insect_folder, file_name)

        # ข้ามถ้าไฟล์มีอยู่แล้ว (เผื่อรันใหม่รอบสอง)
        if os.path.exists(file_path):
            continue

        try:
            # ดาวน์โหลดรูป
            response = requests.get(image_url, timeout=10)
            if response.status_code == 200:
                with open(file_path, 'wb') as f:
                    f.write(response.content)
                if index % 10 == 0:
                    print(f"ดาวน์โหลดสำเร็จ ({index}): {file_name}")
            else:
                print(f"ไม่สามารถโหลดได้ ({response.status_code}): {image_url}")
        except Exception as e:
            print(f"เกิดข้อผิดพลาดที่แถว {index}: {e}")
        # break

# 5. รันฟังก์ชัน
download_images(df, base_download_path)
print("--- เสร็จสิ้นภารกิจดาวน์โหลด ---")

# SECTION 8: trains 3 Models with 1600 images

In [ ]:
import os
import shutil
import random

def split_insect_data(source_path, train_path, val_path, split_ratio=0.8):
    """
    source_path: โฟลเดอร์ต้นทางที่มีโฟลเดอร์ย่อยของแมลงแต่ละชนิด
    train_path: โฟลเดอร์ปลายทางสำหรับ Training
    val_path: โฟลเดอร์ปลายทางสำหรับ Validation
    split_ratio: สัดส่วนการแบ่ง (0.8 คือ Train 80%, Val 20%)
    """

    # 1. ดึงรายชื่อโฟลเดอร์แมลง (Categories)
    categories = [d for d in os.listdir(source_path)
                  if os.path.isdir(os.path.join(source_path, d))]

    for category in categories:
        # สร้างโฟลเดอร์ปลายทางสำหรับแต่ละชนิดแมลง
        os.makedirs(os.path.join(train_path, category), exist_ok=True)
        os.makedirs(os.path.join(val_path, category), exist_ok=True)

        # 2. ดึงรายชื่อไฟล์รูปภาพทั้งหมดในโฟลเดอร์นั้นๆ
        cat_src_dir = os.path.join(source_path, category)
        images = [f for f in os.listdir(cat_src_dir)
                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        # 3. สุ่มลำดับรูปภาพ
        random.shuffle(images)

        # 4. คำนวณจุดตัด (Split point)
        split_point = int(len(images) * split_ratio)
        train_images = images[:split_point]
        val_images = images[split_point:]

        # 5. คัดลอกไฟล์ไปยังปลายทาง
        # ใช้ shutil.copy แทน move เพื่อป้องกันข้อมูลต้นฉบับหาย (เผื่อเทรนพลาด)
        for img in train_images:
            shutil.copy(os.path.join(cat_src_dir, img),
                        os.path.join(train_path, category, img))

        for img in val_images:
            shutil.copy(os.path.join(cat_src_dir, img),
                        os.path.join(val_path, category, img))

        print(f"✅ {category}: แบ่งเรียบร้อย (Train: {len(train_images)}, Val: {len(val_images)})")

# --- วิธีการเรียกใช้งาน ---
# SOURCE_DIR = 'my_original_insects'  # ชื่อโฟลเดอร์ที่มีแมลง 20 ชนิด
SOURCE_DIR = '/content/drive/MyDrive/Colab Notebooks/insect-data/downloaded_images'  # ชื่อโฟลเดอร์ที่มีแมลง 20 ชนิด
TRAIN_DIR = '/content/drive/MyDrive/Colab Notebooks/insect-data/dataset2/train'
VAL_DIR = '/content/drive/MyDrive/Colab Notebooks/insect-data/dataset2/validation'

split_insect_data(SOURCE_DIR, TRAIN_DIR, VAL_DIR, split_ratio=0.8)
print("\n--- แยกข้อมูลเสร็จสิ้น พร้อมนำไปเทรนแล้วครับ! ---")

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras import regularizers
import json
import pandas as pd
import matplotlib.pyplot as plt

# 1. การตั้งค่าพารามิเตอร์ (ใช้เหมือนกันทุกโมเดลเพื่อความยุติธรรม)
IMG_SIZE = (224, 224)
BATCH_SIZE = 8
EPOCHS = 50
# NUM_CLASSES = 22
NUM_CLASSES = 16
LEARNING_RATE = 1e-4 # equals to 0.0001

# 2. ฟังก์ชันสร้างโมเดลแบบ Dynamic
def build_comparison_model(model_name):
    input_shape = (224, 224, 3)

    if model_name == 'MobileNetV2':
        base_model = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
        preprocess = tf.keras.applications.mobilenet_v2.preprocess_input
    elif model_name == 'EfficientNetB0':
        base_model = tf.keras.applications.EfficientNetB0(input_shape=input_shape, include_top=False, weights='imagenet')
        preprocess = tf.keras.applications.efficientnet.preprocess_input
    elif model_name == 'ResNet50V2':
        base_model = tf.keras.applications.ResNet50V2(input_shape=input_shape, include_top=False, weights='imagenet')
        preprocess = tf.keras.applications.resnet_v2.preprocess_input

    base_model.trainable = False # Freeze ส่วนฐานไว้

    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Lambda(preprocess), # จัดการ Preprocess อัตโนมัติในตัวโมเดล
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    model.compile(optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# 3. เริ่มการทดลองเปรียบเทียบ

# โหลดข้อมูลจากโฟลเดอร์
train_datagen = ImageDataGenerator()
train_generator = train_datagen.flow_from_directory(
    '/content/drive/MyDrive/Colab Notebooks/insect-data/dataset2/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_datagen = ImageDataGenerator()
val_generator = val_datagen.flow_from_directory(
    '/content/drive/MyDrive/Colab Notebooks/insect-data/dataset2/validation',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

model_names = ['MobileNetV2', 'EfficientNetB0', 'ResNet50V2']
model_names = ['EfficientNetB0', 'ResNet50V2']
model_names = ['ResNet50V2']
results = []

for name in model_names:
    print(f"\n--- กำลังเทรนโมเดล: {name} ---")
    model = build_comparison_model(name)

    # หมายเหตุ: train_generator และ val_generator ต้องถูกสร้างไว้ก่อนแล้ว
    history = model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=val_generator,
        verbose=1
    )

    # เก็บค่าที่ดีที่สุดของการเทรนครั้งนั้น
    max_train_acc = max(history.history['accuracy'])
    max_val_acc = max(history.history['val_accuracy'])
    min_val_loss = min(history.history['val_loss'])

    results.append({
        'Model': name,
        'Max Train Acc': max_train_acc,
        'Max Val Acc': max_val_acc,
        'Min Val Loss': min_val_loss
    })

    # 7. ดึงข้อมูลจาก history.history (ซึ่งเป็น Dictionary อยู่แล้ว) และบันทึกลงไฟล์ .json
    history_dict = history.history
    with open('/content/drive/MyDrive/Colab Notebooks/insect-data/dataset2/'+name+'_train_history.json', 'w', encoding='utf-8') as f:
        json.dump(history_dict, f, ensure_ascii=False, indent=4)
    break

# 4. แสดงผลลัพธ์ในรูปแบบตาราง (DataFrame)
# df_results = pd.DataFrame(results)
# print("\n--- ตารางสรุปผลการเปรียบเทียบ ---")
# print(df_results)

# เซฟตารางเป็นไฟล์ CSV เพื่อเอาไปใช้ใน Excel หรือใส่เล่มวิจัย
# df_results.to_csv('/content/drive/MyDrive/Colab Notebooks/insect-data/dataset/model_comparison_results.csv', index=False)

In [ ]:
import json
import matplotlib.pyplot as plt
import os
import pandas as pd # เพิ่ม import pandas

# รายชื่อโมเดลตามที่คุณตั้งไว้
model_names = ['MobileNetV2', 'EfficientNetB0', 'ResNet50V2']
path_prefix = '/content/drive/MyDrive/Colab Notebooks/insect-data/dataset2'

plt.figure(figsize=(16, 6))

# 1. กราฟฝั่ง Accuracy
plt.subplot(1, 2, 1)
for name in model_names:
    file_path = os.path.join(path_prefix, f'{name}_train_history.json')

    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            history = json.load(f)

        epochs = range(1, len(history['accuracy']) + 1)
        plt.plot(epochs, history['val_accuracy'], label=f'{name} (Val Acc)', linewidth=2)
    else:
        print(f"⚠️ ไม่พบไฟล์: {file_path}")

plt.title('Validation Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# 2. กราฟฝั่ง Loss
plt.subplot(1, 2, 2)
for name in model_names:
    file_path = os.path.join(path_prefix, f'{name}_train_history.json')

    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            history = json.load(f)

        epochs = range(1, len(history['loss']) + 1)
        plt.plot(epochs, history['val_loss'], label=f'{name} (Val Loss)', linewidth=2)

plt.title('Validation Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig(os.path.join(path_prefix, 'comparison_plot.png'))
plt.show()

# --- ส่วนเพิ่มสำหรับ Export ข้อมูลเป็น CSV ---
export_data = []

for name in model_names:
    file_path = os.path.join(path_prefix, f'{name}_train_history.json')

    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            history = json.load(f)

        epochs = range(1, len(history['val_accuracy']) + 1)
        # สร้าง dictionary สำหรับแต่ละโมเดลและแต่ละ epoch
        for i, epoch in enumerate(epochs):
            export_data.append({
                'Model': name,
                'Epoch': epoch,
                'Validation Accuracy': history['val_accuracy'][i],
                'Validation Loss': history['val_loss'][i]
            })
    else:
        print(f"⚠️ ไม่พบไฟล์ history สำหรับ Export: {file_path}")

if export_data:
    df_export = pd.DataFrame(export_data)
    csv_output_path = os.path.join(path_prefix, 'chart_comparison_data.csv')
    df_export.to_csv(csv_output_path, index=False)
    print(f"\n✅ ข้อมูลจาก Chart ได้ถูก Export เป็น CSV เรียบร้อยแล้วที่: {csv_output_path}")
    display(df_export.head()) # แสดง 5 แถวแรกของ DataFrame ที่ Export
else:
    print("\n❗ ไม่มีข้อมูลสำหรับ Export เป็น CSV (อาจไม่พบไฟล์ history)")
